In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/merged_flight_weather.csv"

df = pd.read_csv(file_path, low_memory=False)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 1065161
Columns: 43


,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),...,rain_flag,thunderstorm_flag,freezing_rain_flag,snow_flag,fog_flag,rain_intensity,thunderstorm_intensity,freezing_rain_intensity,snow_intensity,fog_intensity
0,AS,01/01/2016,15.0,N552AS,SEA,18:45,19:10,400.0,373.0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AS,01/01/2016,25.0,N492AS,SEA,07:00,07:03,405.0,371.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AS,01/01/2016,33.0,N549AS,PDX,17:10,17:20,399.0,332.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AS,01/01/2016,769.0,N526AS,SAN,18:15,18:12,419.0,370.0,-3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AS,01/01/2017,15.0,N568AS,SEA,18:50,21:40,400.0,384.0,170.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
weather_flags = [
    "rain_flag",
    "snow_flag",
    "fog_flag",
    "thunderstorm_flag",
    "freezing_rain_flag"
]

df[weather_flags] = df[weather_flags].fillna(0)

In [ ]:
df[weather_flags].isna().sum()

,0
rain_flag,0
snow_flag,0
fog_flag,0
thunderstorm_flag,0
freezing_rain_flag,0


In [ ]:
feature_cols = [
    "Carrier Name",
    "Season",
    "Day_Name",
    "Time_Bucket",
    "Scheduled_Hour",
    "Scheduled elapsed time (Minutes)",
    "rain_flag",
    "snow_flag",
    "fog_flag",
    "thunderstorm_flag",
    "freezing_rain_flag"
]

In [ ]:
X = df[feature_cols]
y = df["Delay_20"]

print(X.shape)
print(y.shape)
X.head()

(1065161, 11)
(1065161,)


,Carrier Name,Season,Day_Name,Time_Bucket,Scheduled_Hour,Scheduled elapsed time (Minutes),rain_flag,snow_flag,fog_flag,thunderstorm_flag,freezing_rain_flag
0,Alaska Airlines,Winter,Friday,Evening,18,400.0,0.0,0.0,0.0,0.0,0.0
1,Alaska Airlines,Winter,Friday,Early Morning,7,405.0,0.0,0.0,0.0,0.0,0.0
2,Alaska Airlines,Winter,Friday,Afternoon,17,399.0,0.0,0.0,0.0,0.0,0.0
3,Alaska Airlines,Winter,Friday,Evening,18,419.0,0.0,0.0,0.0,0.0,0.0
4,Alaska Airlines,Winter,Sunday,Evening,18,400.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
X = pd.get_dummies(X, drop_first=True)

print(X.shape)
X.head()

(1065161, 28)


,Scheduled_Hour,Scheduled elapsed time (Minutes),rain_flag,snow_flag,fog_flag,thunderstorm_flag,freezing_rain_flag,Carrier Name_American Airlines,Carrier Name_Delta Air Lines,Carrier Name_Frontier Airlines,...,Day_Name_Saturday,Day_Name_Sunday,Day_Name_Thursday,Day_Name_Tuesday,Day_Name_Wednesday,Time_Bucket_Early Morning,Time_Bucket_Evening,Time_Bucket_Late Night,Time_Bucket_Midday,Time_Bucket_Morning Peak
0,18,400.0,0.0,0.0,0.0,0.0,0.0,False,False,False,...,False,False,False,False,False,False,True,False,False,False
1,7,405.0,0.0,0.0,0.0,0.0,0.0,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2,17,399.0,0.0,0.0,0.0,0.0,0.0,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,18,419.0,0.0,0.0,0.0,0.0,0.0,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,18,400.0,0.0,0.0,0.0,0.0,0.0,False,False,False,...,False,True,False,False,False,False,True,False,False,False


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (852128, 28) (852128,)
Test: (213033, 28) (213033,)


In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import roc_auc_score

y_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)

print("AUC:", round(auc, 3))

AUC: 0.706


In [ ]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=False)

importances_sorted.head(15)

,0
Time_Bucket_Evening,0.342808
Time_Bucket_Early Morning,0.224428
Scheduled_Hour,0.094125
fog_flag,0.056756
Time_Bucket_Morning Peak,0.049100
Season_Summer,0.035156
Carrier Name_JetBlue Airways,0.025074
rain_flag,0.019524
Carrier Name_Spirit Airlines,0.015743
Carrier Name_Frontier Airlines,0.014827


In [ ]:
intensity_map = {
    "light": 1,
    "moderate": 2,
    "heavy": 3
}

intensity_cols = [
    "rain_intensity",
    "snow_intensity",
    "fog_intensity",
    "thunderstorm_intensity",
    "freezing_rain_intensity"
]

for col in intensity_cols:
    df[col + "_num"] = df[col].map(intensity_map)

intensity_num_cols = [col + "_num" for col in intensity_cols]
df[intensity_num_cols] = df[intensity_num_cols].fillna(0)

In [ ]:
feature_cols_intensity = [
    "Carrier Name",
    "Season",
    "Day_Name",
    "Time_Bucket",
    "Scheduled_Hour",
    "Scheduled elapsed time (Minutes)",

    # flags
    "rain_flag",
    "snow_flag",
    "fog_flag",
    "thunderstorm_flag",
    "freezing_rain_flag",

    # intensity
    "rain_intensity_num",
    "snow_intensity_num",
    "fog_intensity_num",
    "thunderstorm_intensity_num",
    "freezing_rain_intensity_num"
]

In [ ]:
X_int = df[feature_cols_intensity]
y_int = df["Delay_20"]

In [ ]:
X_int = pd.get_dummies(X_int, drop_first=True)

In [ ]:
X_train_int, X_test_int, y_train_int, y_test_int = train_test_split(
    X_int, y_int,
    test_size=0.2,
    random_state=42,
    stratify=y_int
)

In [ ]:
model.fit(X_train_int, y_train_int)

from sklearn.metrics import roc_auc_score

y_proba_int = model.predict_proba(X_test_int)[:, 1]
auc_int = roc_auc_score(y_test_int, y_proba_int)

print("AUC with intensity:", round(auc_int, 3))

AUC with intensity: 0.706


In [ ]:
import pandas as pd

importances_int = pd.Series(model.feature_importances_, index=X_int.columns)
importances_int_sorted = importances_int.sort_values(ascending=False)

importances_int_sorted.head(20)

,0
Time_Bucket_Evening,0.270597
Time_Bucket_Early Morning,0.209273
Time_Bucket_Morning Peak,0.113893
Scheduled_Hour,0.087047
fog_flag,0.049997
Season_Summer,0.036038
fog_intensity_num,0.023682
Carrier Name_JetBlue Airways,0.023304
rain_flag,0.018470
Time_Bucket_Midday,0.015925
